In [5]:
from src.data.load_data import load_courses
from src.features.bow_features import build_course_bow
from src.features.course_similarity import (
    create_bow_matrix,
    calculate_course_similarity,
    get_similar_courses,
)

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data.load_data import (
    load_course_genre,
    load_ratings,
    load_user_profiles,
)
from src.recommenders.content_based import UserProfileRecommender

DATA_DIR = PROJECT_ROOT / "data" / "raw"

In [2]:
course_genre = load_course_genre(DATA_DIR)
ratings = load_ratings(DATA_DIR)
user_profiles = load_user_profiles(DATA_DIR)

print("Datasets loaded.")
print(f"Courses: {len(course_genre):,}")
print(f"Users: {len(user_profiles):,}")
print(f"Ratings: {len(ratings):,}")

Datasets loaded.
Courses: 307
Users: 33,901
Ratings: 233,306


In [3]:
user_profile_recommender = UserProfileRecommender(top_n=10)

user_profile_recommender.fit(
    user_profile=user_profiles,
    course_genre=course_genre,
)

print("User-profile recommender fitted successfully.")

User-profile recommender fitted successfully.


In [4]:
user_id = int(user_profiles["user"].iloc[0])

recommendations = user_profile_recommender.recommend(
    user_id=user_id,
    course_genre=course_genre,
    ratings=ratings,
)

recommendations

,COURSE_ID,TITLE,score
0,excourse73,analyzing big data with sql,136.0
1,excourse72,foundations for big data analysis with sql,136.0
2,GPXX0IBEN,data science in insurance basic statistical a...,105.0
3,GPXX0TY1EN,performing database operations in the cloudant...,101.0
4,excourse31,cloud computing applications part 2 big data...,99.0
5,excourse02,business intelligence and data warehousing,95.0
6,GPXX0M6UEN,using the cql shell to execute keyspace operat...,93.0
7,GPXX097UEN,performing table and crud operations with cass...,93.0
8,excourse10,database architecture scale and nosql with e...,93.0
9,excourse03,nosql systems,93.0


In [7]:
courses = load_courses(DATA_DIR)

print(f"Courses loaded: {len(courses):,}")

Courses loaded: 307


In [9]:
courses_bow = build_course_bow(courses)

print(f"BoW records: {len(courses_bow):,}")

courses_bow.head()

BoW records: 10,923


,doc_id,token,bow
0,ML0201EN,ai,2
1,ML0201EN,along,1
2,ML0201EN,apps,2
3,ML0201EN,build,2
4,ML0201EN,cloud,1


In [10]:
courses_bow.to_csv(
    PROJECT_ROOT / "data" / "processed" / "courses_bows.csv",
    index=False,
)

print("Saved: data/processed/courses_bows.csv")

Saved: data/processed/courses_bows.csv


In [11]:
bow_matrix = create_bow_matrix(courses_bow)

print(f"BoW matrix shape: {bow_matrix.shape}")

bow_matrix.head()

BoW matrix shape: (307, 2815)


token,ability,able,abstraction,aca,academia,academic,accelerate,accelerated,accelerating,acceleration,...,younes,youracclaim,zara,zeppelin,zero,zillow,zipkin,zombies,zookeeper,zoom
doc_id,,,,,,,,,,,,,,,,,,,,,
AI0111EN,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BC0101EN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BC0201EN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BC0202EN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BD0101EN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
course_similarity = calculate_course_similarity(bow_matrix)

print(f"Similarity matrix shape: {course_similarity.shape}")

course_similarity.iloc[:5, :5]

Similarity matrix shape: (307, 307)


,AI0111EN,BC0101EN,BC0201EN,BC0202EN,BD0101EN
AI0111EN,1.000000,0.044281,0.028583,0.024254,0.041595
BC0101EN,0.044281,1.000000,0.387298,0.182574,0.031311
BC0201EN,0.028583,0.387298,1.000000,0.259272,0.020211
BC0202EN,0.024254,0.182574,0.259272,1.000000,0.017150
BD0101EN,0.041595,0.031311,0.020211,0.017150,1.000000


In [13]:
course_similarity.to_csv(
    PROJECT_ROOT / "data" / "processed" / "sim.csv"
)

print("Saved: data/processed/sim.csv")

Saved: data/processed/sim.csv


In [14]:
target_course = "ML0101ENv3"

similar_courses = get_similar_courses(
    course_id=target_course,
    similarity_matrix=course_similarity,
    courses=courses,
    top_n=10,
)

similar_courses

,COURSE_ID,TITLE,similarity
0,ML0151EN,machine learning with r,0.643204
1,excourse47,machine learning for all,0.608871
2,excourse46,machine learning,0.570962
3,excourse60,introduction to tensorflow for artificial inte...,0.524452
4,ML0109EN,machine learning dimensionality reduction,0.510527
5,excourse69,machine learning with big data,0.448090
6,excourse51,introduction to machine learning in production,0.444524
7,excourse48,introduction to machine learning language pro...,0.437595
8,ML0101EN,machine learning with python,0.434885
9,excourse61,convolutional neural networks in tensorflow,0.423196


In [16]:
import importlib
import src.recommenders.content_based as content_based

importlib.reload(content_based)

<module 'src.recommenders.content_based' from '/home/Tam-Davies/Projects/ml-course-recommender/src/recommenders/content_based.py'>

In [17]:
from src.recommenders.content_based import (
    recommend_from_course_similarity,
)

In [18]:
course_similarity_recommendations = content_based.recommend_from_course_similarity(
    user_id=user_id,
    ratings=ratings,
    similarity_matrix=course_similarity,
    courses=courses,
    top_n=10,
)

course_similarity_recommendations

,COURSE_ID,TITLE,similarity
0,excourse32,introduction to data analytics,0.255327
1,excourse68,big data modeling and management systems,0.243609
2,excourse74,fundamentals of big data,0.238561
3,excourse67,introduction to big data,0.233593
4,excourse36,data analysis using python,0.232874
5,excourse23,data analysis using python,0.232874
6,excourse33,excel basics for data analysis,0.231159
7,excourse69,machine learning with big data,0.225657
8,excourse04,\nsql for data science,0.222231
9,excourse38,data analysis with python,0.220760
